In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from inference import MicroLadPredictor
from models import DDPM

In [ ]:
class ZeroTimeUNet(nn.Module):
    def forward(self, z_t, t):
        return torch.zeros_like(z_t)


class UpsampleVAE(nn.Module):
    def encode(self, x):
        mu = F.avg_pool2d(x, kernel_size=4, stride=4).repeat(1, 4, 1, 1)
        return mu, torch.zeros_like(mu)

    def decode(self, z):
        return F.interpolate(z[:, :1], scale_factor=4, mode="nearest").clamp(0, 1)

In [ ]:
condition = torch.ones(1, 1, 128, 128)
predictor = MicroLadPredictor(
    vae=UpsampleVAE(), unet=ZeroTimeUNet(), ddpm=DDPM(timesteps=1)
)
result = predictor.predict(
    {"size": 128, "images": [{"image": condition, "axis": 0, "index": 32}]}
)

print(result["volume"].shape)
assert result["volume"].shape == torch.Size([128, 1, 128, 128])
assert torch.allclose(result["volume"][32], torch.ones(1, 128, 128))